# 06 - Camada Gold: Modelagem Dimensional, KPIs Analíticos e Destino Duplo
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 3 - Camada Gold):
1. **Consumo Estrito Silver-to-Gold:** Leitura das tabelas Delta higienizadas da Camada Silver no namespace isolado `squad2/grupo1/silver/`. Zero leitura de arquivos da camada `raw`.
2. **Modelagem Dimensional (Data Marts):**
   * **`gold_dim_produtos`:** Dimensão desnormalizada com categorização hierárquica (categoria pai/filha), faixas de precificação e status de disponibilidade comercial.
   * **`gold_metricas_categorias`:** Agregações analíticas e KPIs executivos por categoria (volumetria de SKUs ativos/inativos, taxa de disponibilidade, preço médio/mínimo/máximo, dispersão e monitoramento de categorias raiz).
3. **Auditoria e Integridade Matemática:**
   * Garantia de consistência estrita para dashboards e apresentação ao PO (Vinícius Silveira).
   * Fechamento contábil: `total_skus_ativos + total_skus_inativos == total_skus_cadastrados`.
4. **Destino Duplo Mandatório (*Dual Sink*):**
   * **ADLS Gen2 (Delta Lake):** Persistência analítica com ACID no caminho `squad2/grupo1/gold/`.
   * **Azure SQL Server (DW Relacional):** Replicação no schema `squad2` (`squad2.gold_dim_produtos` e `squad2.gold_metricas_categorias`) via conector nativo `format("sqlserver")` com fallback automático `pyodbc`.


## 1. Carregamento Seguro das Credenciais e Dependências Serverless
Instalação isolada das bibliotecas no escopo da sessão e injeção granular de `adls_options` OAuth no cluster Spark.


In [0]:
# Instalação das dependências necessárias (escopo da sessão Serverless)
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pyodbc pandas --quiet


In [0]:
dbutils.library.restartPython()


In [0]:
import os
from dotenv import load_dotenv, find_dotenv

# Carregamento automático e resiliente do arquivo .env com busca multi-nível
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert sql_password, "SQL_PASSWORD não configurado no .env"

# Configurações OAuth do Service Principal para injeção granular nas operações do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Namespace oficial isolado da Squad 2 / Dupla 1 (/grupo1/)
base_silver = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1/silver"
base_gold = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1/gold"

caminhos = {
    "silver_produtos": f"{base_silver}/ecommerce_produtos",
    "silver_categorias": f"{base_silver}/ecommerce_categorias",
    "gold_dim_produtos": f"{base_gold}/gold_dim_produtos",
    "gold_metricas_categorias": f"{base_gold}/gold_metricas_categorias"
}

print("Caminhos oficiais configurados para a Camada Gold:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")


## 2. Ingestão Silver-to-Gold (Leitura Estrita de Dados Higienizados)
Consumo direto das tabelas Delta curadas na Silver. Zero leitura de dados brutos da camada raw.


In [0]:
print("Carregando tabelas curadas da Camada Silver (Delta Lake)...")

df_silver_produtos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(caminhos["silver_produtos"])
)

df_silver_categorias = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(caminhos["silver_categorias"])
)

total_prod_silver = df_silver_produtos.count()
total_cat_silver = df_silver_categorias.count()

print(f"  [Silver] ecommerce_produtos carregados:   {total_prod_silver} registros")
print(f"  [Silver] ecommerce_categorias carregadas: {total_cat_silver} registros")


## 3. Construção do Data Mart 1: `gold_dim_produtos`
Tabela dimensional desnormalizada para BI e consumo analítico.
* **Join:** Junção dos produtos higienizados com a taxonomia de categorias (incluindo desnormalização da categoria pai).
* **Campos Calculados:**
  * `status_disponibilidade`: 'Ativo' ou 'Inativo'.
  * `faixa_preco`: Segmentação mercadológica para filtros e análises de distribuição de ticket.
  * `gold_processed_at`: Rastreabilidade de auditoria do processamento analítico.


In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp, round as spark_round, broadcast

# Categorias selecionadas para enriquecimento dimensional (otimizado com broadcast devido à baixa volumetria)
df_cat_lookup = df_silver_categorias.select(
    col("id_categoria"),
    col("nome_categoria"),
    col("id_categoria_pai"),
    col("nome_categoria_pai"),
    col("tipo_categoria")
)

df_gold_dim_produtos = (
    df_silver_produtos
    .join(broadcast(df_cat_lookup), on="id_categoria", how="left")
    .withColumn(
        "status_disponibilidade",
        when(col("is_ativo") == True, lit("Ativo")).otherwise(lit("Inativo"))
    )
    .withColumn(
        "faixa_preco",
        when(col("preco_lista") <= 50.0, lit("1. Até R$ 50"))
        .when(col("preco_lista") <= 200.0, lit("2. R$ 50 a R$ 200"))
        .when(col("preco_lista") <= 1000.0, lit("3. R$ 200 a R$ 1.000"))
        .otherwise(lit("4. Acima de R$ 1.000"))
    )
    .withColumn("gold_processed_at", current_timestamp())
    .select(
        col("sku"),
        col("nome_produto"),
        col("descricao"),
        col("id_categoria"),
        col("nome_categoria"),
        col("id_categoria_pai"),
        col("nome_categoria_pai"),
        col("tipo_categoria"),
        col("nome_marca"),
        col("unidade_medida"),
        spark_round(col("preco_lista"), 2).alias("preco_lista"),
        col("is_ativo"),
        col("status_disponibilidade"),
        col("faixa_preco"),
        col("gold_processed_at")
    )
)

print(f"Tabela gold_dim_produtos gerada com sucesso: {df_gold_dim_produtos.count()} produtos.")
display(df_gold_dim_produtos.limit(5))


## 4. Construção do Data Mart 2: `gold_metricas_categorias`
Agregações executivas para avaliação de sortimento e precificação.
* **Granularidade:** `id_categoria`, `nome_categoria`, `id_categoria_pai`, `nome_categoria_pai`.
* **Métricas Consolidadas:**
  * `is_categoria_raiz`: Indicador booleano de categoria topo de hierarquia.
  * `total_skus_cadastrados`: Volume total de produtos associados à categoria.
  * `total_skus_ativos` e `total_skus_inativos`: Divisão de disponibilidade de estoque.
  * `taxa_disponibilidade_pct`: Percentual de SKUs comercializáveis.
  * `preco_medio_lista`, `preco_minimo_lista`, `preco_maximo_lista` e `amplitude_preco`: Dispersão de valores.
  * `total_marcas_distintas`: Diversidade de fornecedores por categoria.


In [0]:
from pyspark.sql.functions import (
    count, sum as spark_sum, avg as spark_avg, min as spark_min, max as spark_max,
    countDistinct, coalesce
)

# Agrupamento analítico a partir da dimensão de produtos
df_agregado_prod = (
    df_gold_dim_produtos
    .groupBy("id_categoria")
    .agg(
        count("sku").alias("total_skus_cadastrados"),
        spark_sum(when(col("is_ativo") == True, 1).otherwise(0)).alias("total_skus_ativos"),
        spark_sum(when(col("is_ativo") == False, 1).otherwise(0)).alias("total_skus_inativos"),
        spark_round(spark_avg("preco_lista"), 2).alias("preco_medio_lista"),
        spark_round(spark_min("preco_lista"), 2).alias("preco_minimo_lista"),
        spark_round(spark_max("preco_lista"), 2).alias("preco_maximo_lista"),
        countDistinct("nome_marca").alias("total_marcas_distintas")
    )
    .withColumn(
        "amplitude_preco",
        spark_round(col("preco_maximo_lista") - col("preco_minimo_lista"), 2)
    )
    .withColumn(
        "taxa_disponibilidade_pct",
        when(
            col("total_skus_cadastrados") > 0,
            spark_round((col("total_skus_ativos") / col("total_skus_cadastrados")) * 100.0, 2)
        ).otherwise(lit(0.0))
    )
)

# Junção com todas as categorias da base para incluir categorias sem produtos ativos (preservação de taxonomia completa)
df_gold_metricas_categorias = (
    df_silver_categorias
    .join(df_agregado_prod, on="id_categoria", how="left")
    .withColumn("is_categoria_raiz", col("id_categoria_pai").isNull())
    .withColumn("total_skus_cadastrados", coalesce(col("total_skus_cadastrados"), lit(0)))
    .withColumn("total_skus_ativos", coalesce(col("total_skus_ativos"), lit(0)))
    .withColumn("total_skus_inativos", coalesce(col("total_skus_inativos"), lit(0)))
    .withColumn("total_marcas_distintas", coalesce(col("total_marcas_distintas"), lit(0)))
    .withColumn("taxa_disponibilidade_pct", coalesce(col("taxa_disponibilidade_pct"), lit(0.0)))
    .withColumn("gold_calculated_at", current_timestamp())
    .select(
        col("id_categoria"),
        col("nome_categoria"),
        col("id_categoria_pai"),
        col("nome_categoria_pai"),
        col("tipo_categoria"),
        col("is_categoria_raiz"),
        col("total_skus_cadastrados"),
        col("total_skus_ativos"),
        col("total_skus_inativos"),
        col("taxa_disponibilidade_pct"),
        col("preco_medio_lista"),
        col("preco_minimo_lista"),
        col("preco_maximo_lista"),
        col("amplitude_preco"),
        col("total_marcas_distintas"),
        col("gold_calculated_at")
    )
)

print(f"Tabela gold_metricas_categorias gerada com sucesso: {df_gold_metricas_categorias.count()} categorias agregadas.")
display(df_gold_metricas_categorias.orderBy(col("total_skus_cadastrados").desc()).limit(10))


## 5. Auditoria de Data Quality e Fechamento Matemático Estrito
Validação mandatória para garantir consistência contábil na entrega ao PO e Tech Lead:
* **Fechamento 1:** `total_skus_ativos + total_skus_inativos == total_skus_cadastrados` em 100% dos registros.
* **Fechamento 2:** A soma dos SKUs agregados na tabela de categorias deve coincidir com o total de SKUs da tabela de produtos.
* **Integridade de Chaves:** 0 nulos nas chaves primárias (`sku` e `id_categoria`).


In [0]:
# 1. Checagem de Fechamento Contábil na Tabela de Métricas
inconsistencias_soma = (
    df_gold_metricas_categorias
    .filter((col("total_skus_ativos") + col("total_skus_inativos")) != col("total_skus_cadastrados"))
    .count()
)
assert inconsistencias_soma == 0, f"Erro de consistência: {inconsistencias_soma} linhas com divergência matemática!"
print("  [DQ PASS] Fechamento Contábil: Total Ativos + Total Inativos == Total Cadastrados (100% íntegro).")

# 2. Checagem de Totalizador Geral
soma_skus_categorias = df_gold_metricas_categorias.select(spark_sum("total_skus_cadastrados")).first()[0]
total_produtos_dim = df_gold_dim_produtos.count()
assert soma_skus_categorias == total_produtos_dim, f"Divergência entre SKUs da dimensão ({total_produtos_dim}) e categorias ({soma_skus_categorias})!"
print(f"  [DQ PASS] Total Geral Conciliado: {total_produtos_dim} SKUs em ambas as visões.")

# 3. Checagem de Nulos nas Chaves Primárias
nulos_sku = df_gold_dim_produtos.filter(col("sku").isNull()).count()
nulos_cat = df_gold_metricas_categorias.filter(col("id_categoria").isNull()).count()
assert nulos_sku == 0 and nulos_cat == 0, "Nulos identificados em chaves primárias na Gold!"
print("  [DQ PASS] Chaves Primárias: 0 nulos em 'sku' e 0 nulos em 'id_categoria'.")

# 4. Alerta de Categorias Raiz (Regra de Negócio 2)
total_raizes = df_gold_metricas_categorias.filter(col("is_categoria_raiz") == True).count()
print(f"  [Governança] Total de Categorias Raiz identificadas: {total_raizes}")


## 6. Persistência Dupla Oficial (*Dual Sink*)
Conforme diretriz obrigatória do Tech Lead Geovany Câncio:
1. **Sink 1 (Data Lake):** Persistência analítica Delta Lake no namespace isolado `squad2/grupo1/gold/`.
2. **Sink 2 (Azure SQL Server):** Persistência no Data Warehouse no schema `squad2` via conector nativo `format("sqlserver")` com fallback estratégico `pyodbc`.


In [0]:
# ==============================================================================
# SINK 1: Persistência no Azure Data Lake Storage Gen2 (Delta Lake ACID)
# ==============================================================================
print("Iniciando gravação no ADLS Gen2 (Delta Lake)...")

(
    df_gold_dim_produtos.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(caminhos["gold_dim_produtos"])
)
print(f"  -> Salvo com sucesso: {caminhos['gold_dim_produtos']}")

(
    df_gold_metricas_categorias.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(caminhos["gold_metricas_categorias"])
)
print(f"  -> Salvo com sucesso: {caminhos['gold_metricas_categorias']}")


In [0]:
# ==============================================================================
# SINK 2: Persistência no Azure SQL Server (DW Analítico)
# ==============================================================================
def salvar_no_sql_server(df, nome_tabela: str):
    """
    Grava o DataFrame na tabela relacional do schema squad2 no Azure SQL Server.
    Utiliza conector nativo sqlserver do Databricks com fallback resiliente para pyodbc.
    """
    print(f"Iniciando persistência no SQL Server na tabela '{nome_tabela}'...")
    
    # 1. Tentativa via conector nativo "sqlserver"
    try:
        (
            df.write
            .format("sqlserver")
            .option("host", sql_host)
            .option("port", "1433")
            .option("user", sql_user)
            .option("password", sql_password)
            .option("database", sql_database)
            .option("dbtable", nome_tabela)
            .mode("overwrite")
            .save()
        )
        print(f"  [Sucesso] Tabela '{nome_tabela}' gravada via conector nativo 'sqlserver'!")
        return
    except Exception as e_sql:
        print(f"  Aviso: Conector 'sqlserver' direto retornou: {e_sql}")

    # 2. Fallback resiliente via pyodbc (Bypass homologado para Databricks Serverless)
    print("  -> Ativando fallback resiliente via pyodbc...")
    import pyodbc
    import pandas as pd
    pdf = df.toPandas()
    
    conn_str = (
        f"DRIVER={{ODBC Driver 18 for SQL Server}};"
        f"SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    try:
        conn = pyodbc.connect(conn_str, timeout=30)
    except Exception:
        conn_str_legacy = (
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};"
            "Encrypt=yes;TrustServerCertificate=no;"
        )
        conn = pyodbc.connect(conn_str_legacy, timeout=30)
        
    cursor = conn.cursor()
    cursor.fast_executemany = True
    
    # Mapeamento de tipos simplificado para criação DDL
    type_map = {
        'object': 'NVARCHAR(MAX)',
        'int64': 'BIGINT',
        'int32': 'INT',
        'float64': 'FLOAT',
        'bool': 'BIT',
        'datetime64[ns]': 'DATETIME2'
    }
    
    cols_def = []
    for col_name, dtype in pdf.dtypes.items():
        col_type = type_map.get(str(dtype), 'NVARCHAR(MAX)')
        if col_name == 'sku':
            col_type = 'VARCHAR(60) NOT NULL'
        elif col_name == 'id_categoria':
            col_type = 'INT NOT NULL'
        cols_def.append(f"[{col_name}] {col_type}")
        
    schema_name, table_raw = nome_tabela.split('.') if '.' in nome_tabela else ('squad2', nome_tabela)
    
    cursor.execute(f"IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = '{schema_name}') EXEC('CREATE SCHEMA [{schema_name}]')")
    cursor.execute(f"IF OBJECT_ID('[{schema_name}].[{table_raw}]', 'U') IS NOT NULL DROP TABLE [{schema_name}].[{table_raw}]")
    cursor.execute(f"CREATE TABLE [{schema_name}].[{table_raw}] ({', '.join(cols_def)})")
    conn.commit()
    
    # Inserção em batch
    placeholders = ', '.join(['?'] * len(pdf.columns))
    insert_sql = f"INSERT INTO [{schema_name}].[{table_raw}] VALUES ({placeholders})"
    records = [
        tuple(None if pd.isna(val) else (int(val) if isinstance(val, bool) else val) for val in row)
        for row in pdf.values
    ]
    cursor.executemany(insert_sql, records)
    conn.commit()
    cursor.close()
    conn.close()
    print(f"  [Sucesso] Tabela '{nome_tabela}' gravada com sucesso via fallback pyodbc ({len(pdf)} linhas)!")

# Execução da persistência no SQL Server para ambas as tabelas Gold
salvar_no_sql_server(df_gold_dim_produtos, "squad2.gold_dim_produtos")
salvar_no_sql_server(df_gold_metricas_categorias, "squad2.gold_metricas_categorias")


## 7. Sumário Executivo da Camada Gold para o Dashboard / PO
Resumo dos indicadores consolidados para apresentação executiva ao cliente (PO Vinícius):


In [0]:
total_skus = df_gold_dim_produtos.count()
total_ativos = df_gold_dim_produtos.filter(col("is_ativo") == True).count()
total_inativos = df_gold_dim_produtos.filter(col("is_ativo") == False).count()
taxa_disponibilidade = round((total_ativos / total_skus) * 100.0, 2)
total_categorias = df_gold_metricas_categorias.count()
ticket_medio_geral = round(df_gold_dim_produtos.select(spark_avg("preco_lista")).first()[0], 2)

print("=" * 70)
print("        RELATÓRIO EXECUTIVO DA CAMADA GOLD — SQUAD 2 DUPLA 1")
print("=" * 70)
print(f"  • Total de SKUs Únicos no Catálogo:      {total_skus:,}")
print(f"  • SKUs Ativos (Comercializáveis):        {total_ativos:,} ({taxa_disponibilidade}%)")
print(f"  • SKUs Inativos (Fora de Linha/Pausa):   {total_inativos:,}")
print(f"  • Total de Categorias Estruturadas:      {total_categorias}")
print(f"  • Preço Médio Global do Portfólio:       R$ {ticket_medio_geral:,.2f}")
print("=" * 70)
print("Status do Pipeline Gold: 100% CONCLUÍDO E HOMOLOGADO!")
